---

# MISSING DATA REPRESENTATION IN PANDAS

---


`Missing Data` in Pandas is often represented by NumPy `NaN` values

    - This is more efficient than Python's `None` data type
    - Pandas treats `NaN` values as a float, which allows them to be used in vectorized operations

---

In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = [0, 5, 155, np.nan, 518]\

sales_series = pd.Series(data=sales, name='Sales')
sales_series

0      0.0
1      5.0
2    155.0
3      NaN
4    518.0
Name: Sales, dtype: float64

---

## `NaN` values don't change with mathematical operations

In [3]:
sales_series + 2

0      2.0
1      7.0
2    157.0
3      NaN
4    520.0
Name: Sales, dtype: float64

In [4]:
sales_series.mul(2)

0       0.0
1      10.0
2     310.0
3       NaN
4    1036.0
Name: Sales, dtype: float64

## Nevertheless we can assing a value to `NaN` values

In [5]:
sales_series.add(3, fill_value = 0)

0      3.0
1      8.0
2    158.0
3      3.0
4    521.0
Name: Sales, dtype: float64

---

Pandas released its own **missing values data type**, `NA`, in December 2020.

    - This allows missing values to be stored as integers, instead of needing to convert to float.
    - This is still a new feature, but most bugs end up converting the data to NumPy's NaN.

---

In [6]:
sales = [0, 5, 155, pd.NA, 518]

sales_series = pd.Series(data=sales, name='sales', dtype='Int16')
sales_series

0       0
1       5
2     155
3    <NA>
4     518
Name: sales, dtype: Int16

---

At this time, **neither `np.nan` or `pd.NA` are perfect**, but `pd.NA` functionality should continue to improve, and having a nullable integer is usually worth it.

---

---

## Personal Laboratory `np.array` VS. `np.ndarray`

---

- `np.ndarray` es la clase (el tipo de objeto). 

- `np.array` es una función fábrica (factory function) que crea instancias de esa clase.

Es decir: cuando tú escribes `np.array([1, 2, 3])`, estás llamando a una función que internamente construye y te devuelve un objeto cuyo tipo es `np.ndarray`. Veámoslo:

In [7]:
import numpy as np

arr = np.array([1, 2, 3])
print(f'type(arr):          {type(arr)}')        # <class 'numpy.ndarray'>
print(f'type(np.array):     {type(np.array)}')   # <class 'builtin_function_or_method'>
print(f'type(np.ndarray):   {type(np.ndarray)}') # <class 'type'>   ← es una clase

type(arr):          <class 'numpy.ndarray'>
type(np.array):     <class 'builtin_function_or_method'>
type(np.ndarray):   <class 'type'>


<br>

---

Fíjate en esto: 

- `type(np.ndarray)` devuelve **type**, lo cual significa "esto es una clase". 
- `type(np.array)` devuelve **builtin_function_or_method**, lo cual significa "esto es una función".
- `type(arr)` devuelve **numpy.ndarray**, confirmando que el objeto resultante es una instancia de la clase.

<br><br>

### El "por qué" del diseño

>Esta es la parte interesante, porque conecta con un patrón muy común en Python que vas a ver una y otra vez.
>
>En teoría, podrías construir un `ndarray` directamente llamando al constructor de la clase:


In [8]:
arr = np.ndarray(shape=(3,), dtype=int)
print(type(arr))
arr

<class 'numpy.ndarray'>


array([97662137125323,              0,             32])


>Pero si lo pruebas, vas a ver algo raro: el array se crea con valores basura (memoria sin inicializar).
> 
>Esto es porque el constructor de `ndarray es una API de bajo nivel` — espera que tú le digas exactamente cuánta memoria reservar, con qué tipo, con qué strides, etc.
>
>No infiere nada por ti. Está pensado para uso interno de NumPy y desarrolladores avanzados.
>
>`np.array()`, en cambio, es una `API de alto nivel diseñada para humanos`: le pasas una lista de Python, una tupla, otro array, lo que sea, y la función se encarga de:
>
>- Inferir la forma (shape) automáticamente
>- Inferir el tipo de dato (dtype) más apropiado
>- Copiar los datos a memoria nueva
>- Validar la estructura (que no sea irregular)
>- Devolverte un ndarray listo para usar
>
>Es el clásico patrón de "constructor amigable" que oculta complejidad. En lugar de obligarte a entender el modelo de memoria de NumPy para crear un array trivial, te ofrece una puerta de entrada limpia.
>
<br><br>

### La conexión con OOP que te interesa

>Este patrón se llama `factory function` (función fábrica), y es fundamental en Python. Cuando aprendas a diseñar tus propias clases más adelante (OOP), vas a usar este patrón muchas veces. La idea es:
>
>- La `clase` define qué es el objeto y cómo se comporta.
>- La `factory function` se encarga de construirlo con buenos defaults a partir de inputs convenientes para el usuario.
>
>Vas a ver esto en **Scikit-Learn**, en **PyTorch**, en casi todas las librerías serias. Por ejemplo, en **PyTorch** existe `torch.Tensor` (la clase) y `torch.tensor()` (la factory function) — exactamente la misma dualidad. La convención informal es: clase con inicial mayúscula, función fábrica con inicial minúscula.

<br><br>

---

### Otras funciones fábrica que ya conoces (sin saberlo)
>`np.array()` no es la única función que produce ndarrays. Todas estas también lo hacen:

In [9]:
np.zeros((3, 3))        # ndarray de ceros
np.ones((3, 3))         # ndarray de unos
np.empty((3, 3))        # ndarray sin inicializar (rápido pero peligroso)
np.arange(10)           # ndarray con secuencia
np.linspace(0, 1, 11)   # ndarray con valores equiespaciados
np.full((3, 3), 7)      # ndarray relleno con un valor 7
np.random.rand(3, 3)    # ndarray aleatorio

# Todas comparten:
type(np.zeros((3, 3))) == type(np.array([1, 2, 3]))  # True → ambos son ndarray

True

> Todas estas son `factory functions` especializadas. Cada una resuelve un caso de uso común sin que tengas que pasar por la `API genérica de np.array()`.

<br><br>

---

### La intuición que te quiero dejar

>Cuando leas documentación de NumPy y veas escrito "returns: ndarray", ya sabes que se refiere al tipo del objeto devuelto. Cuando escribas código, vas a llamar a `np.array()` (u otra fábrica) porque es la forma idiomática de crear arrays. Y cuando hagas type hints rigurosos en tu código (Estandard de código), vas a escribir:

In [10]:
def normalizar(x: np.ndarray) -> np.ndarray:
    """Normaliza un array a media 0 y desviación estándar 1."""
    return (x - x.mean()) / x.std()

>Nota que el `type hint` usa `np.ndarray` (la clase), no `np.array` (la función). Esto es importante: los type hints anotan tipos, y `np.array` no es un tipo, es una función.

---

**Una regla mnemotécnica simple:**

>`clase = sustantivo` (qué es). `ndarray` es el sustantivo (un array N-dimensional).
>
>`función = verbo` (qué hace). `array(...)` es el verbo (la acción de crear uno).